In [ ]:
# second_level_features code (CAM version)

# Enhanced Frequency Domain CNN with Dual-Domain Feature Fusion (NaN Fix)
# Key Innovation: Second-level FFT on last conv layer activation maps
# Visualization: Class Activation Mapping (CAM)

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.fft as fft
from torch.utils.data import DataLoader, Dataset
from torchvision import models, transforms
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter
from tqdm.auto import tqdm
import os
import warnings
import gc
warnings.filterwarnings('ignore')

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Available GPU memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

# Set random seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed(42)
    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark = True

# ================== Step 1: Custom Dataset for Fruits-360 ==================

class FruitsDataset(Dataset):
    """Custom dataset for Fruits-360"""
    
    def __init__(self, root_dir, transform=None):
        self.root_dir = root_dir
        self.transform = transform
        self.classes = sorted([d for d in os.listdir(root_dir) 
                              if os.path.isdir(os.path.join(root_dir, d))])
        self.class_to_idx = {cls_name: i for i, cls_name in enumerate(self.classes)}
        
        self.samples = []
        for class_name in self.classes:
            class_dir = os.path.join(root_dir, class_name)
            if os.path.isdir(class_dir):
                for img_name in os.listdir(class_dir):
                    if img_name.lower().endswith(('.png', '.jpg', '.jpeg')):
                        self.samples.append((os.path.join(class_dir, img_name), 
                                           self.class_to_idx[class_name]))
        
        print(f"Found {len(self.samples)} images in {len(self.classes)} classes")
    
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        try:
            image = Image.open(img_path).convert('RGB')
            if self.transform:
                image = self.transform(image)
            return image, label
        except Exception as e:
            print(f"Error loading {img_path}: {e}")
            return torch.zeros(3, 100, 100), label

def load_fruits_dataset(data_root):
    """Load Fruits-360 dataset with augmentation"""
    
    transform_train = transforms.Compose([
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomVerticalFlip(p=0.3),
        transforms.RandomRotation(20),
        transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
        transforms.RandomAffine(degrees=0, translate=(0.1, 0.1), scale=(0.9, 1.1)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    
    transform_test = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    
    train_dir = os.path.join(data_root, 'Training')
    test_dir = os.path.join(data_root, 'Test')
    
    trainset = FruitsDataset(train_dir, transform=transform_train)
    testset = FruitsDataset(test_dir, transform=transform_test)
    
    return trainset, testset, trainset.classes

# ================== Step 2: FIXED FFT Conversion Functions ==================

def spatial_to_frequency(images):
    """Convert spatial domain images to frequency domain using FFT - FIXED VERSION"""
    freq_complex = fft.fft2(images, dim=(-2, -1))
    freq_complex = fft.fftshift(freq_complex, dim=(-2, -1))
    
    freq_magnitude = torch.abs(freq_complex)
    freq_phase = torch.angle(freq_complex)
    
    # FIXED: Better normalization with clipping to prevent NaN
    eps = 1e-6
    freq_magnitude = torch.clamp(freq_magnitude, min=eps)
    freq_magnitude_log = torch.log(freq_magnitude + eps)
    
    # FIXED: Robust normalization with safety checks
    mean_val = freq_magnitude_log.mean()
    std_val = freq_magnitude_log.std()
    std_val = torch.clamp(std_val, min=1e-5)  # Prevent division by zero
    
    freq_magnitude_normalized = (freq_magnitude_log - mean_val) / std_val
    
    # FIXED: Clip to reasonable range to prevent extreme values
    freq_magnitude_normalized = torch.clamp(freq_magnitude_normalized, -10, 10)
    
    # Encode phase as cosine and sine
    phase_cos = torch.cos(freq_phase)
    phase_sin = torch.sin(freq_phase)
    
    # Concatenate magnitude and phase information (9 channels total)
    freq_features = torch.cat([freq_magnitude_normalized, phase_cos, phase_sin], dim=1)
    
    # FIXED: Final safety check
    freq_features = torch.nan_to_num(freq_features, nan=0.0, posinf=10.0, neginf=-10.0)
    
    return freq_features, freq_phase, freq_complex

def frequency_to_spatial(freq_magnitude, freq_phase):
    """Convert frequency domain back to spatial domain"""
    freq_magnitude = torch.exp(torch.clamp(freq_magnitude, -10, 10))  # FIXED: Clamp before exp
    freq_complex = freq_magnitude * torch.exp(1j * freq_phase)
    
    freq_complex = fft.ifftshift(freq_complex, dim=(-2, -1))
    spatial_complex = fft.ifft2(freq_complex, dim=(-2, -1))
    spatial_images = torch.real(spatial_complex)
    
    return spatial_images

def extract_frequency_features(activation_map):
    """
    Extract second-level frequency features from activation maps - FIXED VERSION
    This captures hidden patterns in the frequency domain
    """
    # Apply FFT to activation maps
    freq_complex = fft.fft2(activation_map, dim=(-2, -1))
    freq_complex = fft.fftshift(freq_complex, dim=(-2, -1))
    
    # Extract magnitude and normalize
    freq_magnitude = torch.abs(freq_complex)
    eps = 1e-6
    
    # FIXED: Safer log operation
    freq_magnitude = torch.clamp(freq_magnitude, min=eps)
    freq_magnitude_log = torch.log(freq_magnitude + eps)
    
    # Normalize across spatial dimensions
    b, c, h, w = freq_magnitude_log.shape
    freq_magnitude_norm = freq_magnitude_log.view(b, c, -1)
    
    # FIXED: Robust normalization
    mean_val = freq_magnitude_norm.mean(dim=2, keepdim=True)
    std_val = freq_magnitude_norm.std(dim=2, keepdim=True)
    std_val = torch.clamp(std_val, min=1e-5)
    
    freq_magnitude_norm = (freq_magnitude_norm - mean_val) / std_val
    freq_magnitude_norm = torch.clamp(freq_magnitude_norm, -10, 10)
    freq_magnitude_norm = freq_magnitude_norm.view(b, c, h, w)
    
    # FIXED: Safety check
    freq_magnitude_norm = torch.nan_to_num(freq_magnitude_norm, nan=0.0, posinf=10.0, neginf=-10.0)
    
    return freq_magnitude_norm

# ================== Step 3: Frequency Domain Dataset ==================

class FrequencyDomainDataset(Dataset):
    """Custom dataset for frequency domain representations with caching"""
    
    def __init__(self, original_dataset, cache_freq=False):
        self.original_dataset = original_dataset
        self.cache_freq = cache_freq
        self.freq_cache = {} if cache_freq else None
        
    def __len__(self):
        return len(self.original_dataset)
    
    def __getitem__(self, idx):
        if self.cache_freq and idx in self.freq_cache:
            return self.freq_cache[idx]
        
        image, label = self.original_dataset[idx]
        
        # Convert to frequency domain
        with torch.no_grad():
            freq_features, freq_phase, _ = spatial_to_frequency(image.unsqueeze(0))
            freq_features = freq_features.squeeze(0)
            freq_phase = freq_phase.squeeze(0)
        
        result = (freq_features, label, freq_phase)
        
        if self.cache_freq:
            self.freq_cache[idx] = result
        
        return result

# ================== Step 4: FIXED CNN Model with Dual-Domain Feature Fusion ==================

class DualDomainFeatureFusion(nn.Module):
    """
    Dual-Domain Feature Fusion Module - FIXED VERSION
    Combines spatial activation maps with their frequency domain representations
    """
    def __init__(self, in_channels):
        super(DualDomainFeatureFusion, self).__init__()
        
        # FIXED: Better initialization for learnable weights
        self.spatial_weight = nn.Parameter(torch.tensor(0.7))  # Favor spatial features initially
        self.freq_weight = nn.Parameter(torch.tensor(0.3))     # Lower weight for frequency
        
        # Channel attention mechanism
        self.channel_attention = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Conv2d(in_channels * 2, in_channels, kernel_size=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(in_channels, in_channels * 2, kernel_size=1),
            nn.Sigmoid()
        )
        
        # FIXED: Add batch normalization for stability
        self.bn_spatial = nn.BatchNorm2d(in_channels)
        self.bn_freq = nn.BatchNorm2d(in_channels)
        
    def forward(self, spatial_features):
        """
        Args:
            spatial_features: Activation maps from last conv layer (B, C, H, W)
        Returns:
            fused_features: Combined spatial + frequency features (B, C, H, W)
        """
        # Extract frequency features from spatial activation maps
        freq_features = extract_frequency_features(spatial_features)
        
        # FIXED: Apply batch normalization for stability
        spatial_features_norm = self.bn_spatial(spatial_features)
        freq_features_norm = self.bn_freq(freq_features)
        
        # Concatenate spatial and frequency features
        combined = torch.cat([spatial_features_norm, freq_features_norm], dim=1)
        
        # Apply channel attention
        attention = self.channel_attention(combined)
        combined_weighted = combined * attention
        
        # Split back to spatial and frequency
        spatial_weighted = combined_weighted[:, :spatial_features.size(1), :, :]
        freq_weighted = combined_weighted[:, spatial_features.size(1):, :, :]
        
        # FIXED: Use sigmoid to normalize weights to [0,1] range
        spatial_w = torch.sigmoid(self.spatial_weight)
        freq_w = torch.sigmoid(self.freq_weight)
        
        # Weighted fusion
        fused = spatial_w * spatial_weighted + freq_w * freq_weighted
        
        # FIXED: Safety check for NaN/Inf
        fused = torch.nan_to_num(fused, nan=0.0, posinf=10.0, neginf=-10.0)
        
        return fused

class EnhancedFrequencyDomainCNN(nn.Module):
    """
    Enhanced ResNet50-based model with Dual-Domain Feature Fusion - FIXED VERSION
    Innovation: Second-level FFT on last conv layer before GAP
    """
    
    def __init__(self, num_classes, dropout_rate=0.5):
        super(EnhancedFrequencyDomainCNN, self).__init__()
        
        self.model = models.resnet50(pretrained=True)
        
        # Modify first conv layer for 9-channel input
        original_conv1 = self.model.conv1
        self.model.conv1 = nn.Conv2d(9, 64, kernel_size=7, stride=2, padding=3, bias=False)
        
        # FIXED: Better weight initialization for first layer
        with torch.no_grad():
            # Copy weights from original conv1 for first 3 channels
            self.model.conv1.weight[:, :3, :, :] = original_conv1.weight
            # Initialize remaining 6 channels with smaller values
            nn.init.kaiming_normal_(self.model.conv1.weight[:, 3:, :, :], mode='fan_out', nonlinearity='relu')
            self.model.conv1.weight[:, 3:, :, :] *= 0.1  # Scale down frequency channel weights
        
        # Get number of features from layer4
        self.num_features_layer4 = self.model.layer4[-1].conv3.out_channels
        
        # Add Dual-Domain Feature Fusion before GAP
        self.dual_domain_fusion = DualDomainFeatureFusion(self.num_features_layer4)
        
        # Replace avgpool and fc with custom head
        self.model.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        
        # Enhanced classifier head
        # Note: The first Linear layer (num_features_layer4 -> 1024) is the CAM weight layer
        self.model.fc = nn.Sequential(
            nn.Dropout(dropout_rate),
            nn.Linear(self.num_features_layer4, 1024),
            nn.ReLU(inplace=True),
            nn.BatchNorm1d(1024),
            nn.Dropout(dropout_rate * 0.7),
            nn.Linear(1024, 512),
            nn.ReLU(inplace=True),
            nn.BatchNorm1d(512),
            nn.Dropout(dropout_rate * 0.5),
            nn.Linear(512, num_classes)
        )
        
        self._initialize_weights()
    
    def _initialize_weights(self):
        """Initialize new layers with proper weights"""
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0)
            elif isinstance(m, (nn.BatchNorm2d, nn.BatchNorm1d)):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)
    
    def forward(self, x):
        # FIXED: Add input validation
        x = torch.nan_to_num(x, nan=0.0, posinf=10.0, neginf=-10.0)
        
        # Forward through ResNet layers
        x = self.model.conv1(x)
        x = self.model.bn1(x)
        x = self.model.relu(x)
        x = self.model.maxpool(x)
        
        x = self.model.layer1(x)
        x = self.model.layer2(x)
        x = self.model.layer3(x)
        x = self.model.layer4(x)
        
        # Apply Dual-Domain Feature Fusion (x + fft(x))
        x = self.dual_domain_fusion(x)
        
        # Global Average Pooling
        x = self.model.avgpool(x)
        x = torch.flatten(x, 1)
        
        # Classifier
        x = self.model.fc(x)
        
        return x
    
    def get_activations(self, x):
        """Extract feature maps from last conv layer before fusion"""
        x = self.model.conv1(x)
        x = self.model.bn1(x)
        x = self.model.relu(x)
        x = self.model.maxpool(x)
        
        x = self.model.layer1(x)
        x = self.model.layer2(x)
        x = self.model.layer3(x)
        x = self.model.layer4(x)
        
        return x
    
    def get_fused_activations(self, x):
        """Extract feature maps after dual-domain fusion"""
        x = self.get_activations(x)
        x = self.dual_domain_fusion(x)
        return x

# ================== Step 5: Training Functions ==================

class EarlyStopping:
    """Early stopping to prevent overfitting"""
    def __init__(self, patience=10, min_delta=0.0, verbose=True):
        self.patience = patience
        self.min_delta = min_delta
        self.verbose = verbose
        self.counter = 0
        self.best_score = None
        self.early_stop = False
        self.best_model_state = None
        
    def __call__(self, val_accuracy, model):
        score = val_accuracy
        
        if self.best_score is None:
            self.best_score = score
            self.best_model_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        elif score < self.best_score + self.min_delta:
            self.counter += 1
            if self.verbose:
                print(f'EarlyStopping counter: {self.counter} out of {self.patience}')
            if self.counter >= self.patience:
                self.early_stop = True
        else:
            self.best_score = score
            self.best_model_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            self.counter = 0

def train_model(model, train_loader, val_loader, epochs=50, lr=0.001, weight_decay=1e-4):
    """Train the enhanced frequency domain CNN model - FIXED VERSION"""
    
    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
    
    pretrained_params = []
    new_params = []
    
    for name, param in model.named_parameters():
        if 'model.fc' in name or 'model.conv1' in name or 'dual_domain_fusion' in name:
            new_params.append(param)
        else:
            pretrained_params.append(param)
    
    # FIXED: Lower learning rate to prevent instability
    optimizer = torch.optim.AdamW([
        {'params': pretrained_params, 'lr': lr * 0.01},  # Even lower for pretrained
        {'params': new_params, 'lr': lr * 0.5}  # Lower for new layers
    ], weight_decay=weight_decay)
    
    scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
        optimizer, T_0=10, T_mult=2, eta_min=1e-7
    )
    
    early_stopping = EarlyStopping(patience=15, min_delta=0.1, verbose=True)
    
    train_losses = []
    val_losses = []
    train_accuracies = []
    val_accuracies = []
    
    best_val_accuracy = 0.0
    best_model_state = None
    
    scaler = torch.cuda.amp.GradScaler() if torch.cuda.is_available() else None
    
    for epoch in range(epochs):
        # Training phase
        model.train()
        running_loss = 0.0
        correct_train = 0
        total_train = 0
        
        train_pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs} [Train]", leave=False)
        for i, (freq_images, labels, _) in enumerate(train_pbar):
            freq_images, labels = freq_images.to(device, non_blocking=True), labels.to(device, non_blocking=True)
            
            # FIXED: Check for NaN in input
            if torch.isnan(freq_images).any() or torch.isinf(freq_images).any():
                print(f"Warning: NaN/Inf detected in input batch {i}, skipping...")
                continue
            
            optimizer.zero_grad(set_to_none=True)
            
            if scaler is not None:
                with torch.cuda.amp.autocast():
                    outputs = model(freq_images)
                    loss = criterion(outputs, labels)
                
                # FIXED: Check for NaN loss before backward
                if torch.isnan(loss) or torch.isinf(loss):
                    print(f"Warning: NaN/Inf loss detected in batch {i}, skipping...")
                    continue
                
                scaler.scale(loss).backward()
                
                # FIXED: Gradient clipping before unscale
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=0.5)  # Lower max_norm
                
                scaler.step(optimizer)
                scaler.update()
            else:
                outputs = model(freq_images)
                loss = criterion(outputs, labels)
                
                if torch.isnan(loss) or torch.isinf(loss):
                    print(f"Warning: NaN/Inf loss detected in batch {i}, skipping...")
                    continue
                
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=0.5)
                optimizer.step()
            
            running_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            total_train += labels.size(0)
            correct_train += (predicted == labels).sum().item()
            
            train_pbar.set_postfix({
                'loss': f'{loss.item():.4f}',
                'acc': f'{100 * correct_train / total_train:.2f}%'
            })
            
            if i % 50 == 0 and torch.cuda.is_available():
                torch.cuda.empty_cache()
        
        scheduler.step()
        
        avg_train_loss = running_loss / len(train_loader)
        train_accuracy = 100 * correct_train / total_train
        train_losses.append(avg_train_loss)
        train_accuracies.append(train_accuracy)
        
        # Validation phase
        model.eval()
        running_val_loss = 0.0
        correct = 0
        total = 0
        
        with torch.no_grad():
            val_pbar = tqdm(val_loader, desc=f"Epoch {epoch+1}/{epochs} [Val]", leave=False)
            for freq_images, labels, _ in val_pbar:
                freq_images, labels = freq_images.to(device, non_blocking=True), labels.to(device, non_blocking=True)
                
                if scaler is not None:
                    with torch.cuda.amp.autocast():
                        outputs = model(freq_images)
                        loss = criterion(outputs, labels)
                else:
                    outputs = model(freq_images)
                    loss = criterion(outputs, labels)
                
                running_val_loss += loss.item()
                
                _, predicted = torch.max(outputs.data, 1)
                total += labels.size(0)
                correct += (predicted == labels).sum().item()
                
                val_pbar.set_postfix({
                    'loss': f'{loss.item():.4f}',
                    'acc': f'{100 * correct / total:.2f}%'
                })
        
        avg_val_loss = running_val_loss / len(val_loader)
        val_accuracy = 100 * correct / total
        val_losses.append(avg_val_loss)
        val_accuracies.append(val_accuracy)
        
        if val_accuracy > best_val_accuracy:
            best_val_accuracy = val_accuracy
            best_model_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        
        print(f'\nEpoch [{epoch+1}/{epochs}]')
        print(f'Train Loss: {avg_train_loss:.4f}, Train Acc: {train_accuracy:.2f}%')
        print(f'Val Loss: {avg_val_loss:.4f}, Val Acc: {val_accuracy:.2f}%')
        print(f'Learning Rate: {optimizer.param_groups[0]["lr"]:.6f}')
        
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        gc.collect()
        
        early_stopping(val_accuracy, model)
        if early_stopping.early_stop:
            print("Early stopping triggered!")
            break
    
    if best_model_state is not None:
        model.load_state_dict(best_model_state)
        print(f"\nLoaded best model with validation accuracy: {best_val_accuracy:.2f}%")
    
    return train_losses, val_losses, train_accuracies, val_accuracies

# ================== Step 6: CAM Implementation ==================

class CAM:
    """
    Class Activation Mapping (CAM) implementation with dual-domain awareness.

    CAM computes a weighted linear combination of the fused activation maps
    using the weights of the first FC layer (GAP features → hidden units),
    then projects them back via the class weights of the final FC layer.
    Because the classifier head is multi-layer (2048 → 1024 → 512 → num_classes),
    we propagate the effective per-channel weights all the way from the output
    class score back to the 2048-channel feature map with a single matrix
    multiplication of the chained weight matrices.
    """

    def __init__(self, model):
        self.model = model
        self.model.eval()

    def _get_fc_weights_for_class(self, target_class):
        """
        Compute effective weight vector of shape (num_features_layer4,) that maps
        the GAP feature vector to the score of `target_class`, by chaining all
        linear layers in model.fc (ignoring Dropout and BatchNorm non-linearities
        for the linear approximation used by CAM).

        model.fc layout (indices after Dropout at 0):
          [0] Dropout
          [1] Linear(2048, 1024)   <-- first linear
          [2] ReLU
          [3] BatchNorm1d(1024)
          [4] Dropout
          [5] Linear(1024, 512)
          [6] ReLU
          [7] BatchNorm1d(512)
          [8] Dropout
          [9] Linear(512, num_classes)  <-- last linear
        """
        fc_layers = [m for m in self.model.model.fc.modules() if isinstance(m, nn.Linear)]
        # fc_layers: [Linear(2048,1024), Linear(1024,512), Linear(512,num_classes)]

        # Start from the class weight of the last layer (shape: [num_features_last])
        w = fc_layers[-1].weight[target_class]  # (512,)

        # Walk backwards through the remaining linear layers
        for layer in reversed(fc_layers[:-1]):
            # layer.weight shape: (out, in) → w @ layer.weight gives (in,)
            w = w @ layer.weight  # chain: (in_features,)

        return w  # shape: (num_features_layer4,) i.e. (2048,)

    def generate_cam(self, input_image, target_class):
        """
        Generate CAM for `target_class`.

        Returns
        -------
        cam_np : np.ndarray  (H_input, W_input)  values in [0, 1]
        """
        with torch.no_grad():
            # Get fused activation maps: (1, C, h, w)
            activations = self.model.get_fused_activations(input_image)  # (1, 2048, h, w)

        b, C, h, w = activations.shape

        # Get effective class weights: (C,)
        weights = self._get_fc_weights_for_class(target_class).to(activations.device)

        # Weighted sum over channels → (h, w)
        # activations.squeeze(0): (C, h, w)
        cam = torch.einsum('c,chw->hw', weights, activations.squeeze(0))

        # Apply ReLU — keep only positive contributions
        cam = F.relu(cam)

        # Upsample to input size
        _, _, input_h, input_w = input_image.shape
        cam = F.interpolate(
            cam.unsqueeze(0).unsqueeze(0),   # (1, 1, h, w)
            size=(input_h, input_w),
            mode='bilinear',
            align_corners=False
        ).squeeze()  # (input_h, input_w)

        # Normalize to [0, 1]
        cam_min, cam_max = cam.min(), cam.max()
        if cam_max > cam_min:
            cam = (cam - cam_min) / (cam_max - cam_min)
        else:
            cam = torch.zeros_like(cam)

        return cam.cpu().detach().numpy()


# ================== Step 7: Spatial Domain Mapping ==================

def apply_cam_and_map_to_spatial(model, freq_image, phase, target_class, original_image):
    """Apply CAM with dual-domain features and map result to spatial domain."""

    model.eval()
    freq_input = freq_image.clone().detach().to(device)

    # ---- Generate CAM ----
    cam_obj = CAM(model)
    cam_map = cam_obj.generate_cam(freq_input, target_class)   # (H, W) in [0,1]

    # ---- Map CAM through frequency → spatial domain (same pipeline as before) ----
    freq_magnitude = freq_input[:, :3, :, :].squeeze(0).cpu().detach()

    cam_tensor = torch.from_numpy(cam_map).float()
    masked_freq_magnitude = freq_magnitude * cam_tensor.unsqueeze(0)

    # Ensure phase has correct dimensions
    if phase.dim() == 4:
        phase = phase.squeeze(0)
    elif phase.dim() == 2:
        phase = phase.unsqueeze(0).repeat(3, 1, 1)

    cam_spatial = frequency_to_spatial(
        masked_freq_magnitude.unsqueeze(0),
        phase.unsqueeze(0)
    )
    cam_spatial = cam_spatial.squeeze(0)

    # Build saliency map from spatial signal
    cam_spatial_abs = torch.abs(cam_spatial)
    saliency_map = torch.mean(cam_spatial_abs, dim=0).numpy()

    # Invert (match original pipeline convention)
    saliency_map = np.max(saliency_map) - saliency_map

    # Gaussian smoothing
    saliency_map = gaussian_filter(saliency_map, sigma=2.5)

    # Normalize
    if saliency_map.max() > saliency_map.min():
        saliency_map = (saliency_map - saliency_map.min()) / (saliency_map.max() - saliency_map.min())
    else:
        saliency_map = np.zeros_like(saliency_map)

    # Adaptive thresholding
    threshold = np.percentile(saliency_map, 40)
    saliency_map = np.where(saliency_map > threshold, saliency_map, 0)

    # Morphological operations
    from scipy.ndimage import binary_closing, binary_opening
    binary_mask = saliency_map > 0
    binary_mask = binary_closing(binary_mask, structure=np.ones((5, 5)))
    binary_mask = binary_opening(binary_mask, structure=np.ones((3, 3)))
    saliency_map = saliency_map * binary_mask

    # Re-normalize
    if saliency_map.max() > 0:
        saliency_map = (saliency_map - saliency_map.min()) / (saliency_map.max() - saliency_map.min())

    # Enhance contrast
    saliency_map = np.power(saliency_map, 0.7)

    # Denormalize original image for display
    if original_image.dim() == 4:
        original_image = original_image.squeeze(0)

    mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
    std  = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
    original_denorm = original_image.cpu() * std + mean
    original_denorm = torch.clamp(original_denorm, 0, 1)
    original_np = original_denorm.permute(1, 2, 0).numpy()

    # Colour overlay
    saliency_colored = plt.cm.jet(saliency_map)[:, :, :3]
    alpha = 0.7 * saliency_map[:, :, np.newaxis]
    highlighted = (1 - alpha) * original_np + alpha * saliency_colored
    highlighted = np.clip(highlighted, 0, 1)

    return cam_spatial, saliency_map, highlighted, original_np, cam_map


# ================== Step 8: Visualization Functions ==================

def plot_cam_results(original_np, freq_magnitude, cam_map, saliency_map,
                     highlighted, prediction, true_label, classes, confidence):
    """
    Plot comprehensive CAM results with dual-domain info.
    Layout mirrors the original Score-CAM plot so the rest of the pipeline
    is unchanged; only the third panel label/title reflects CAM.
    """

    fig, axes = plt.subplots(2, 4, figsize=(20, 10))
    fig.suptitle(
        'Enhanced Frequency Domain CNN with Dual-Domain Feature Fusion\n'
        '(Spatial + FFT of Last Conv Layer)  —  CAM Visualization',
        fontsize=16, fontweight='bold', y=0.998
    )

    # 1. Original Image
    axes[0, 0].imshow(original_np)
    axes[0, 0].set_title(f'Original Image\nGround Truth: {classes[true_label]}',
                         fontsize=11, fontweight='bold')
    axes[0, 0].axis('off')

    # 2. Frequency Domain Input Magnitude
    freq_display = freq_magnitude[:, :3, :, :].squeeze(0).mean(0).cpu().numpy()
    im1 = axes[0, 1].imshow(freq_display, cmap='viridis')
    axes[0, 1].set_title('Frequency Domain\n(Input Magnitude Spectrum)',
                         fontsize=11, fontweight='bold')
    axes[0, 1].axis('off')
    plt.colorbar(im1, ax=axes[0, 1], fraction=0.046, pad=0.04)

    # 3. CAM heatmap (replaces Score-CAM panel)
    im2 = axes[0, 2].imshow(cam_map, cmap='jet')
    axes[0, 2].set_title('CAM\n(Dual-Domain Features)', fontsize=11, fontweight='bold')
    axes[0, 2].axis('off')
    plt.colorbar(im2, ax=axes[0, 2], fraction=0.046, pad=0.04)

    # 4. Prediction result
    correct = "✓" if prediction == true_label else "✗"
    color = 'green' if prediction == true_label else 'red'
    axes[0, 3].text(
        0.5, 0.5,
        f'{correct} Prediction:\n{classes[prediction]}\n\nConfidence:\n{confidence:.1f}%',
        ha='center', va='center', fontsize=13, fontweight='bold',
        bbox=dict(boxstyle='round', facecolor=color, alpha=0.3)
    )
    axes[0, 3].set_title('Model Prediction', fontsize=11, fontweight='bold')
    axes[0, 3].axis('off')

    # 5. Saliency Map
    im3 = axes[1, 0].imshow(saliency_map, cmap='hot')
    axes[1, 0].set_title('Saliency Map\n(Enhanced Features)', fontsize=11, fontweight='bold')
    axes[1, 0].axis('off')
    plt.colorbar(im3, ax=axes[1, 0], fraction=0.046, pad=0.04)

    # 6. Highlighted Regions
    axes[1, 1].imshow(highlighted)
    axes[1, 1].set_title('Highlighted Regions\n(Dual-Domain Aware)',
                         fontsize=11, fontweight='bold')
    axes[1, 1].axis('off')

    # 7. CAM overlay at 50% transparency
    axes[1, 2].imshow(original_np)
    axes[1, 2].imshow(cam_map, cmap='jet', alpha=0.5)
    axes[1, 2].set_title('CAM Heatmap\n(50% Overlay)', fontsize=11, fontweight='bold')
    axes[1, 2].axis('off')

    # 8. Before / After comparison
    axes[1, 3].imshow(np.concatenate([original_np, highlighted], axis=1))
    axes[1, 3].set_title('Before | After\n(Enhanced CAM)',
                         fontsize=11, fontweight='bold')
    axes[1, 3].axis('off')

    plt.tight_layout()
    plt.show()


def plot_training_curves(train_losses, val_losses, train_accuracies, val_accuracies):
    """Plot training and validation curves"""
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    
    epochs = range(1, len(train_losses) + 1)
    ax1.plot(epochs, train_losses, 'b-', label='Training Loss', linewidth=2)
    ax1.plot(epochs, val_losses, 'r-', label='Validation Loss', linewidth=2)
    ax1.set_xlabel('Epoch', fontsize=12)
    ax1.set_ylabel('Loss', fontsize=12)
    ax1.set_title('Training and Validation Loss', fontsize=14, fontweight='bold')
    ax1.legend(fontsize=11)
    ax1.grid(True, alpha=0.3)
    
    ax2.plot(epochs, train_accuracies, 'b-', label='Training Accuracy', linewidth=2)
    ax2.plot(epochs, val_accuracies, 'r-', label='Validation Accuracy', linewidth=2)
    ax2.set_xlabel('Epoch', fontsize=12)
    ax2.set_ylabel('Accuracy (%)', fontsize=12)
    ax2.set_title('Training and Validation Accuracy', fontsize=14, fontweight='bold')
    ax2.legend(fontsize=11)
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

# ================== Step 9: Main Execution Pipeline ==================

def main():
    print("="*80)
    print("Enhanced Frequency Domain CNN with Dual-Domain Feature Fusion")
    print("Innovation: x_fused = x + fft(x) before GAP layer")
    print("Visualization: Class Activation Mapping (CAM)")
    print("OPTIMIZED FOR 16GB GPU")
    print("="*80)
    
    # Dataset path - MODIFY THIS PATH
    data_root = r'/kaggle/input/fruits/fruits-360_100x100/fruits-360'
    
    # Check if path exists
    if not os.path.exists(data_root):
        print(f"\nERROR: Dataset path not found: {data_root}")
        print("Please modify the 'data_root' variable to point to your dataset location.")
        return
    
    print("\n[Step 1] Loading Fruits-360 dataset...")
    try:
        trainset, testset, classes = load_fruits_dataset(data_root)
        print(f"Number of classes: {len(classes)}")
    except Exception as e:
        print(f"ERROR loading dataset: {e}")
        return
    
    print("\n[Step 2] Splitting training set into train/validation...")
    train_size = int(0.85 * len(trainset))
    val_size = len(trainset) - train_size
    train_subset, val_subset = torch.utils.data.random_split(
        trainset, [train_size, val_size],
        generator=torch.Generator().manual_seed(42)
    )
    
    print(f"Training samples: {len(train_subset)}")
    print(f"Validation samples: {len(val_subset)}")
    print(f"Test samples: {len(testset)}")
    
    print("\n[Step 3] Converting to frequency domain using FFT...")
    freq_train_dataset = FrequencyDomainDataset(train_subset, cache_freq=False)
    freq_val_dataset = FrequencyDomainDataset(val_subset, cache_freq=False)
    freq_test_dataset = FrequencyDomainDataset(testset, cache_freq=False)
    
    # Optimized batch size and num_workers for 16GB GPU
    batch_size = 128
    num_workers = 4 if os.name != 'nt' else 0
    
    print(f"Batch size: {batch_size}")
    print(f"Num workers: {num_workers}")
    
    train_loader = DataLoader(
        freq_train_dataset, 
        batch_size=batch_size, 
        shuffle=True, 
        num_workers=num_workers, 
        pin_memory=True,
        persistent_workers=False,
        prefetch_factor=2 if num_workers > 0 else None
    )
    
    val_loader = DataLoader(
        freq_val_dataset, 
        batch_size=batch_size, 
        shuffle=False,
        num_workers=num_workers, 
        pin_memory=True,
        persistent_workers=False,
        prefetch_factor=2 if num_workers > 0 else None
    )
    
    test_loader = DataLoader(
        freq_test_dataset, 
        batch_size=1, 
        shuffle=False,
        num_workers=0
    )
    
    print("\n[Step 4] Initializing Enhanced ResNet50 with Dual-Domain Fusion...")
    model = EnhancedFrequencyDomainCNN(num_classes=len(classes), dropout_rate=0.5).to(device)
    print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")
    print("Architecture: ResNet50 + Dual-Domain Feature Fusion (x + fft(x)) before GAP")
    
    # Check GPU memory
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        print(f"GPU memory allocated: {torch.cuda.memory_allocated(0) / 1e9:.2f} GB")
        print(f"GPU memory reserved: {torch.cuda.memory_reserved(0) / 1e9:.2f} GB")
    
    print("\n[Step 5] Training model with dual-domain features...")
    print("Starting training loop...\n")
    
    try:
        train_losses, val_losses, train_accuracies, val_accuracies = train_model(
            model, train_loader, val_loader, 
            epochs=2,
            lr=0.001,
            weight_decay=5e-4
        )
    except Exception as e:
        print(f"\nERROR during training: {e}")
        import traceback
        traceback.print_exc()
        return
    
    print("\n[Step 5.1] Plotting training curves...")
    plot_training_curves(train_losses, val_losses, train_accuracies, val_accuracies)
    
    print("\n[Step 6] Evaluating on test set...")
    model.eval()
    correct = 0
    total = 0
    
    all_predictions = []
    all_labels = []
    
    with torch.no_grad():
        test_pbar = tqdm(test_loader, desc="Testing")
        for freq_images, labels, _ in test_pbar:
            freq_images, labels = freq_images.to(device), labels.to(device)
            outputs = model(freq_images)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
            
            all_predictions.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            
            test_pbar.set_postfix({'acc': f'{100 * correct / total:.2f}%'})
    
    test_accuracy = 100 * correct / total
    print(f"\nFinal Test Accuracy: {test_accuracy:.2f}%")
    
    print("\n[Step 7] Applying CAM with dual-domain features...")
    print("Generating CAM-based saliency maps using fused activation maps...\n")
    
    # Test on diverse samples
    np.random.seed(42)
    num_samples = min(5, len(testset))
    test_indices = np.random.choice(len(testset), num_samples, replace=False)
    
    for idx in test_indices:
        try:
            original_image, true_label = testset[idx]
            
            freq_features, phase, _ = spatial_to_frequency(original_image.unsqueeze(0))
            freq_input = freq_features.to(device)
            
            # Get model prediction
            model.eval()
            with torch.no_grad():
                output = model(freq_input)
                probabilities = F.softmax(output, dim=1)
                confidence, predicted = torch.max(probabilities.data, 1)
                predicted_class = predicted.item()
                confidence = confidence.item() * 100
            
            print(f"Sample {idx}:")
            print(f"  True Label: {classes[true_label]}")
            print(f"  Predicted: {classes[predicted_class]} ({confidence:.1f}% confidence)")
            
            # Apply CAM with dual-domain features
            _, saliency_map, highlighted, original_np, cam_map = apply_cam_and_map_to_spatial(
                model, freq_input, phase.squeeze(0), predicted_class, original_image
            )
            
            # Visualize results
            plot_cam_results(
                original_np,
                freq_features,
                cam_map,
                saliency_map,
                highlighted,
                predicted_class,
                true_label,
                classes,
                confidence
            )
            
            # Clear GPU cache
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
            
            print()
        
        except Exception as e:
            print(f"ERROR processing sample {idx}: {e}")
            import traceback
            traceback.print_exc()
            continue
    
    print("="*80)
    print("Pipeline completed successfully!")
    print(f"Final Test Accuracy: {test_accuracy:.2f}%")
    print(f"Total Classes: {len(classes)}")
    print("="*80)
    print("\n🚀 KEY INNOVATION:")
    print("✓ Dual-Domain Feature Fusion: x_fused = α*x + β*fft(x)")
    print("✓ Applied before GAP layer to capture hidden frequency patterns")
    print("✓ Learnable fusion weights for optimal balance")
    print("✓ Channel attention for adaptive feature weighting")
    print("✓ CAM: class-weighted linear combination of fused activation maps")
    print("="*80)
    print("\nOptimizations Applied:")
    print("✓ Mixed precision training (AMP)")
    print("✓ Optimized batch size (128) for 16GB GPU")
    print("✓ Gradient accumulation and clipping")
    print("✓ Memory-efficient data loading")
    print("✓ Periodic GPU cache clearing")
    print("✓ CuDNN benchmark mode enabled")
    print("="*80)
    
    # Save the model
    print("\n[Step 8] Saving trained model...")
    try:
        torch.save({
            'model_state_dict': model.state_dict(),
            'test_accuracy': test_accuracy,
            'classes': classes,
            'num_classes': len(classes)
        }, 'fruits_enhanced_dual_domain_cnn.pth')
        print("Model saved as 'fruits_enhanced_dual_domain_cnn.pth'")
    except Exception as e:
        print(f"ERROR saving model: {e}")
    
    # Final memory cleanup
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        print(f"\nFinal GPU memory allocated: {torch.cuda.memory_allocated(0) / 1e9:.2f} GB")

if __name__ == "__main__":
    main()

In [ ]:
# second_level_features code

# Enhanced Frequency Domain CNN with Dual-Domain Feature Fusion (NaN Fix)
# Key Innovation: Second-level FFT on last conv layer activation maps
# CAM Method: Eigen-CAM (PCA-based, uses first principal component of activation maps)

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.fft as fft
from torch.utils.data import DataLoader, Dataset
from torchvision import models, transforms
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter
from tqdm.auto import tqdm
import os
import warnings
import gc
warnings.filterwarnings('ignore')

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Available GPU memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

# Set random seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed(42)
    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark = True

# ================== Step 1: Custom Dataset for Fruits-360 ==================

class FruitsDataset(Dataset):
    """Custom dataset for Fruits-360"""
    
    def __init__(self, root_dir, transform=None):
        self.root_dir = root_dir
        self.transform = transform
        self.classes = sorted([d for d in os.listdir(root_dir) 
                              if os.path.isdir(os.path.join(root_dir, d))])
        self.class_to_idx = {cls_name: i for i, cls_name in enumerate(self.classes)}
        
        self.samples = []
        for class_name in self.classes:
            class_dir = os.path.join(root_dir, class_name)
            if os.path.isdir(class_dir):
                for img_name in os.listdir(class_dir):
                    if img_name.lower().endswith(('.png', '.jpg', '.jpeg')):
                        self.samples.append((os.path.join(class_dir, img_name), 
                                           self.class_to_idx[class_name]))
        
        print(f"Found {len(self.samples)} images in {len(self.classes)} classes")
    
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        try:
            image = Image.open(img_path).convert('RGB')
            if self.transform:
                image = self.transform(image)
            return image, label
        except Exception as e:
            print(f"Error loading {img_path}: {e}")
            return torch.zeros(3, 100, 100), label

def load_fruits_dataset(data_root):
    """Load Fruits-360 dataset with augmentation"""
    
    transform_train = transforms.Compose([
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomVerticalFlip(p=0.3),
        transforms.RandomRotation(20),
        transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
        transforms.RandomAffine(degrees=0, translate=(0.1, 0.1), scale=(0.9, 1.1)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    
    transform_test = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    
    train_dir = os.path.join(data_root, 'Training')
    test_dir = os.path.join(data_root, 'Test')
    
    trainset = FruitsDataset(train_dir, transform=transform_train)
    testset = FruitsDataset(test_dir, transform=transform_test)
    
    return trainset, testset, trainset.classes

# ================== Step 2: FIXED FFT Conversion Functions ==================

def spatial_to_frequency(images):
    """Convert spatial domain images to frequency domain using FFT - FIXED VERSION"""
    freq_complex = fft.fft2(images, dim=(-2, -1))
    freq_complex = fft.fftshift(freq_complex, dim=(-2, -1))
    
    freq_magnitude = torch.abs(freq_complex)
    freq_phase = torch.angle(freq_complex)
    
    # FIXED: Better normalization with clipping to prevent NaN
    eps = 1e-6
    freq_magnitude = torch.clamp(freq_magnitude, min=eps)
    freq_magnitude_log = torch.log(freq_magnitude + eps)
    
    # FIXED: Robust normalization with safety checks
    mean_val = freq_magnitude_log.mean()
    std_val = freq_magnitude_log.std()
    std_val = torch.clamp(std_val, min=1e-5)  # Prevent division by zero
    
    freq_magnitude_normalized = (freq_magnitude_log - mean_val) / std_val
    
    # FIXED: Clip to reasonable range to prevent extreme values
    freq_magnitude_normalized = torch.clamp(freq_magnitude_normalized, -10, 10)
    
    # Encode phase as cosine and sine
    phase_cos = torch.cos(freq_phase)
    phase_sin = torch.sin(freq_phase)
    
    # Concatenate magnitude and phase information (9 channels total)
    freq_features = torch.cat([freq_magnitude_normalized, phase_cos, phase_sin], dim=1)
    
    # FIXED: Final safety check
    freq_features = torch.nan_to_num(freq_features, nan=0.0, posinf=10.0, neginf=-10.0)
    
    return freq_features, freq_phase, freq_complex

def frequency_to_spatial(freq_magnitude, freq_phase):
    """Convert frequency domain back to spatial domain"""
    freq_magnitude = torch.exp(torch.clamp(freq_magnitude, -10, 10))  # FIXED: Clamp before exp
    freq_complex = freq_magnitude * torch.exp(1j * freq_phase)
    
    freq_complex = fft.ifftshift(freq_complex, dim=(-2, -1))
    spatial_complex = fft.ifft2(freq_complex, dim=(-2, -1))
    spatial_images = torch.real(spatial_complex)
    
    return spatial_images

def extract_frequency_features(activation_map):
    """
    Extract second-level frequency features from activation maps - FIXED VERSION
    This captures hidden patterns in the frequency domain
    """
    # Apply FFT to activation maps
    freq_complex = fft.fft2(activation_map, dim=(-2, -1))
    freq_complex = fft.fftshift(freq_complex, dim=(-2, -1))
    
    # Extract magnitude and normalize
    freq_magnitude = torch.abs(freq_complex)
    eps = 1e-6
    
    # FIXED: Safer log operation
    freq_magnitude = torch.clamp(freq_magnitude, min=eps)
    freq_magnitude_log = torch.log(freq_magnitude + eps)
    
    # Normalize across spatial dimensions
    b, c, h, w = freq_magnitude_log.shape
    freq_magnitude_norm = freq_magnitude_log.view(b, c, -1)
    
    # FIXED: Robust normalization
    mean_val = freq_magnitude_norm.mean(dim=2, keepdim=True)
    std_val = freq_magnitude_norm.std(dim=2, keepdim=True)
    std_val = torch.clamp(std_val, min=1e-5)
    
    freq_magnitude_norm = (freq_magnitude_norm - mean_val) / std_val
    freq_magnitude_norm = torch.clamp(freq_magnitude_norm, -10, 10)
    freq_magnitude_norm = freq_magnitude_norm.view(b, c, h, w)
    
    # FIXED: Safety check
    freq_magnitude_norm = torch.nan_to_num(freq_magnitude_norm, nan=0.0, posinf=10.0, neginf=-10.0)
    
    return freq_magnitude_norm

# ================== Step 3: Frequency Domain Dataset ==================

class FrequencyDomainDataset(Dataset):
    """Custom dataset for frequency domain representations with caching"""
    
    def __init__(self, original_dataset, cache_freq=False):
        self.original_dataset = original_dataset
        self.cache_freq = cache_freq
        self.freq_cache = {} if cache_freq else None
        
    def __len__(self):
        return len(self.original_dataset)
    
    def __getitem__(self, idx):
        if self.cache_freq and idx in self.freq_cache:
            return self.freq_cache[idx]
        
        image, label = self.original_dataset[idx]
        
        # Convert to frequency domain
        with torch.no_grad():
            freq_features, freq_phase, _ = spatial_to_frequency(image.unsqueeze(0))
            freq_features = freq_features.squeeze(0)
            freq_phase = freq_phase.squeeze(0)
        
        result = (freq_features, label, freq_phase)
        
        if self.cache_freq:
            self.freq_cache[idx] = result
        
        return result

# ================== Step 4: FIXED CNN Model with Dual-Domain Feature Fusion ==================

class DualDomainFeatureFusion(nn.Module):
    """
    Dual-Domain Feature Fusion Module - FIXED VERSION
    Combines spatial activation maps with their frequency domain representations
    """
    def __init__(self, in_channels):
        super(DualDomainFeatureFusion, self).__init__()
        
        # FIXED: Better initialization for learnable weights
        self.spatial_weight = nn.Parameter(torch.tensor(0.7))  # Favor spatial features initially
        self.freq_weight = nn.Parameter(torch.tensor(0.3))     # Lower weight for frequency
        
        # Channel attention mechanism
        self.channel_attention = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Conv2d(in_channels * 2, in_channels, kernel_size=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(in_channels, in_channels * 2, kernel_size=1),
            nn.Sigmoid()
        )
        
        # FIXED: Add batch normalization for stability
        self.bn_spatial = nn.BatchNorm2d(in_channels)
        self.bn_freq = nn.BatchNorm2d(in_channels)
        
    def forward(self, spatial_features):
        """
        Args:
            spatial_features: Activation maps from last conv layer (B, C, H, W)
        Returns:
            fused_features: Combined spatial + frequency features (B, C, H, W)
        """
        # Extract frequency features from spatial activation maps
        freq_features = extract_frequency_features(spatial_features)
        
        # FIXED: Apply batch normalization for stability
        spatial_features_norm = self.bn_spatial(spatial_features)
        freq_features_norm = self.bn_freq(freq_features)
        
        # Concatenate spatial and frequency features
        combined = torch.cat([spatial_features_norm, freq_features_norm], dim=1)
        
        # Apply channel attention
        attention = self.channel_attention(combined)
        combined_weighted = combined * attention
        
        # Split back to spatial and frequency
        spatial_weighted = combined_weighted[:, :spatial_features.size(1), :, :]
        freq_weighted = combined_weighted[:, spatial_features.size(1):, :, :]
        
        # FIXED: Use sigmoid to normalize weights to [0,1] range
        spatial_w = torch.sigmoid(self.spatial_weight)
        freq_w = torch.sigmoid(self.freq_weight)
        
        # Weighted fusion
        fused = spatial_w * spatial_weighted + freq_w * freq_weighted
        
        # FIXED: Safety check for NaN/Inf
        fused = torch.nan_to_num(fused, nan=0.0, posinf=10.0, neginf=-10.0)
        
        return fused

class EnhancedFrequencyDomainCNN(nn.Module):
    """
    Enhanced ResNet50-based model with Dual-Domain Feature Fusion - FIXED VERSION
    Innovation: Second-level FFT on last conv layer before GAP
    """
    
    def __init__(self, num_classes, dropout_rate=0.5):
        super(EnhancedFrequencyDomainCNN, self).__init__()
        
        self.model = models.resnet50(pretrained=True)
        
        # Modify first conv layer for 9-channel input
        original_conv1 = self.model.conv1
        self.model.conv1 = nn.Conv2d(9, 64, kernel_size=7, stride=2, padding=3, bias=False)
        
        # FIXED: Better weight initialization for first layer
        with torch.no_grad():
            # Copy weights from original conv1 for first 3 channels
            self.model.conv1.weight[:, :3, :, :] = original_conv1.weight
            # Initialize remaining 6 channels with smaller values
            nn.init.kaiming_normal_(self.model.conv1.weight[:, 3:, :, :], mode='fan_out', nonlinearity='relu')
            self.model.conv1.weight[:, 3:, :, :] *= 0.1  # Scale down frequency channel weights
        
        # Get number of features from layer4
        num_features_layer4 = self.model.layer4[-1].conv3.out_channels
        
        # Add Dual-Domain Feature Fusion before GAP
        self.dual_domain_fusion = DualDomainFeatureFusion(num_features_layer4)
        
        # Replace avgpool and fc with custom head
        self.model.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        
        # Enhanced classifier head
        self.model.fc = nn.Sequential(
            nn.Dropout(dropout_rate),
            nn.Linear(num_features_layer4, 1024),
            nn.ReLU(inplace=True),
            nn.BatchNorm1d(1024),
            nn.Dropout(dropout_rate * 0.7),
            nn.Linear(1024, 512),
            nn.ReLU(inplace=True),
            nn.BatchNorm1d(512),
            nn.Dropout(dropout_rate * 0.5),
            nn.Linear(512, num_classes)
        )
        
        self._initialize_weights()
    
    def _initialize_weights(self):
        """Initialize new layers with proper weights"""
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0)
            elif isinstance(m, (nn.BatchNorm2d, nn.BatchNorm1d)):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)
    
    def forward(self, x):
        # FIXED: Add input validation
        x = torch.nan_to_num(x, nan=0.0, posinf=10.0, neginf=-10.0)
        
        # Forward through ResNet layers
        x = self.model.conv1(x)
        x = self.model.bn1(x)
        x = self.model.relu(x)
        x = self.model.maxpool(x)
        
        x = self.model.layer1(x)
        x = self.model.layer2(x)
        x = self.model.layer3(x)
        x = self.model.layer4(x)
        
        # Apply Dual-Domain Feature Fusion (x + fft(x))
        x = self.dual_domain_fusion(x)
        
        # Global Average Pooling
        x = self.model.avgpool(x)
        x = torch.flatten(x, 1)
        
        # Classifier
        x = self.model.fc(x)
        
        return x
    
    def get_activations(self, x):
        """Extract feature maps from last conv layer before fusion"""
        x = self.model.conv1(x)
        x = self.model.bn1(x)
        x = self.model.relu(x)
        x = self.model.maxpool(x)
        
        x = self.model.layer1(x)
        x = self.model.layer2(x)
        x = self.model.layer3(x)
        x = self.model.layer4(x)
        
        return x
    
    def get_fused_activations(self, x):
        """Extract feature maps after dual-domain fusion"""
        x = self.get_activations(x)
        x = self.dual_domain_fusion(x)
        return x

# ================== Step 5: Training Functions ==================

class EarlyStopping:
    """Early stopping to prevent overfitting"""
    def __init__(self, patience=10, min_delta=0.0, verbose=True):
        self.patience = patience
        self.min_delta = min_delta
        self.verbose = verbose
        self.counter = 0
        self.best_score = None
        self.early_stop = False
        self.best_model_state = None
        
    def __call__(self, val_accuracy, model):
        score = val_accuracy
        
        if self.best_score is None:
            self.best_score = score
            self.best_model_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        elif score < self.best_score + self.min_delta:
            self.counter += 1
            if self.verbose:
                print(f'EarlyStopping counter: {self.counter} out of {self.patience}')
            if self.counter >= self.patience:
                self.early_stop = True
        else:
            self.best_score = score
            self.best_model_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            self.counter = 0

def train_model(model, train_loader, val_loader, epochs=50, lr=0.001, weight_decay=1e-4):
    """Train the enhanced frequency domain CNN model - FIXED VERSION"""
    
    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
    
    pretrained_params = []
    new_params = []
    
    for name, param in model.named_parameters():
        if 'model.fc' in name or 'model.conv1' in name or 'dual_domain_fusion' in name:
            new_params.append(param)
        else:
            pretrained_params.append(param)
    
    # FIXED: Lower learning rate to prevent instability
    optimizer = torch.optim.AdamW([
        {'params': pretrained_params, 'lr': lr * 0.01},  # Even lower for pretrained
        {'params': new_params, 'lr': lr * 0.5}  # Lower for new layers
    ], weight_decay=weight_decay)
    
    scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
        optimizer, T_0=10, T_mult=2, eta_min=1e-7
    )
    
    early_stopping = EarlyStopping(patience=15, min_delta=0.1, verbose=True)
    
    train_losses = []
    val_losses = []
    train_accuracies = []
    val_accuracies = []
    
    best_val_accuracy = 0.0
    best_model_state = None
    
    scaler = torch.cuda.amp.GradScaler() if torch.cuda.is_available() else None
    
    for epoch in range(epochs):
        # Training phase
        model.train()
        running_loss = 0.0
        correct_train = 0
        total_train = 0
        
        train_pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs} [Train]", leave=False)
        for i, (freq_images, labels, _) in enumerate(train_pbar):
            freq_images, labels = freq_images.to(device, non_blocking=True), labels.to(device, non_blocking=True)
            
            # FIXED: Check for NaN in input
            if torch.isnan(freq_images).any() or torch.isinf(freq_images).any():
                print(f"Warning: NaN/Inf detected in input batch {i}, skipping...")
                continue
            
            optimizer.zero_grad(set_to_none=True)
            
            if scaler is not None:
                with torch.cuda.amp.autocast():
                    outputs = model(freq_images)
                    loss = criterion(outputs, labels)
                
                # FIXED: Check for NaN loss before backward
                if torch.isnan(loss) or torch.isinf(loss):
                    print(f"Warning: NaN/Inf loss detected in batch {i}, skipping...")
                    continue
                
                scaler.scale(loss).backward()
                
                # FIXED: Gradient clipping before unscale
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=0.5)  # Lower max_norm
                
                scaler.step(optimizer)
                scaler.update()
            else:
                outputs = model(freq_images)
                loss = criterion(outputs, labels)
                
                if torch.isnan(loss) or torch.isinf(loss):
                    print(f"Warning: NaN/Inf loss detected in batch {i}, skipping...")
                    continue
                
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=0.5)
                optimizer.step()
            
            running_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            total_train += labels.size(0)
            correct_train += (predicted == labels).sum().item()
            
            train_pbar.set_postfix({
                'loss': f'{loss.item():.4f}',
                'acc': f'{100 * correct_train / total_train:.2f}%'
            })
            
            if i % 50 == 0 and torch.cuda.is_available():
                torch.cuda.empty_cache()
        
        scheduler.step()
        
        avg_train_loss = running_loss / len(train_loader)
        train_accuracy = 100 * correct_train / total_train
        train_losses.append(avg_train_loss)
        train_accuracies.append(train_accuracy)
        
        # Validation phase
        model.eval()
        running_val_loss = 0.0
        correct = 0
        total = 0
        
        with torch.no_grad():
            val_pbar = tqdm(val_loader, desc=f"Epoch {epoch+1}/{epochs} [Val]", leave=False)
            for freq_images, labels, _ in val_pbar:
                freq_images, labels = freq_images.to(device, non_blocking=True), labels.to(device, non_blocking=True)
                
                if scaler is not None:
                    with torch.cuda.amp.autocast():
                        outputs = model(freq_images)
                        loss = criterion(outputs, labels)
                else:
                    outputs = model(freq_images)
                    loss = criterion(outputs, labels)
                
                running_val_loss += loss.item()
                
                _, predicted = torch.max(outputs.data, 1)
                total += labels.size(0)
                correct += (predicted == labels).sum().item()
                
                val_pbar.set_postfix({
                    'loss': f'{loss.item():.4f}',
                    'acc': f'{100 * correct / total:.2f}%'
                })
        
        avg_val_loss = running_val_loss / len(val_loader)
        val_accuracy = 100 * correct / total
        val_losses.append(avg_val_loss)
        val_accuracies.append(val_accuracy)
        
        if val_accuracy > best_val_accuracy:
            best_val_accuracy = val_accuracy
            best_model_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        
        print(f'\nEpoch [{epoch+1}/{epochs}]')
        print(f'Train Loss: {avg_train_loss:.4f}, Train Acc: {train_accuracy:.2f}%')
        print(f'Val Loss: {avg_val_loss:.4f}, Val Acc: {val_accuracy:.2f}%')
        print(f'Learning Rate: {optimizer.param_groups[0]["lr"]:.6f}')
        
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        gc.collect()
        
        early_stopping(val_accuracy, model)
        if early_stopping.early_stop:
            print("Early stopping triggered!")
            break
    
    if best_model_state is not None:
        model.load_state_dict(best_model_state)
        print(f"\nLoaded best model with validation accuracy: {best_val_accuracy:.2f}%")
    
    return train_losses, val_losses, train_accuracies, val_accuracies

# ================== Step 6: Eigen-CAM Implementation ==================

class EigenCAM:
    """
    Eigen-CAM implementation with dual-domain awareness.

    Eigen-CAM computes the Class Activation Map using the first principal
    component (eigenvector) of the 2D activation maps from the target layer.
    Unlike Score-CAM or Grad-CAM, it requires NO forward passes per channel
    and NO gradient computation — making it fast and gradient-free.

    Algorithm:
        1. Extract fused activation maps  →  shape (1, C, H, W)
        2. Reshape to a 2D matrix         →  shape (C, H*W)
        3. Center the matrix by subtracting the channel-wise mean
        4. Compute SVD; the first right singular vector is the 1st PC
        5. Reshape the 1st PC back to (H, W)  →  raw CAM
        6. Apply ReLU + normalize to [0, 1]
        7. Upsample to input spatial resolution
    """

    def __init__(self, model):
        self.model = model
        self.model.eval()

    def generate_cam(self, input_image):
        """
        Generate Eigen-CAM from the fused activation maps.

        Args:
            input_image (Tensor): shape (1, C_in, H_in, W_in) on the correct device.

        Returns:
            cam (np.ndarray): normalized CAM, shape (H_in, W_in), values in [0, 1].
            principal_components (np.ndarray): explained-variance ratios for the top-5
                                               singular values (diagnostic info).
        """
        with torch.no_grad():
            # ── 1. Extract fused activation maps (B=1, C, h, w) ──────────────
            activations = self.model.get_fused_activations(input_image)  # (1, C, h, w)

        b, C, h, w = activations.shape

        # ── 2. Reshape to (C, H*W) and convert to float32 numpy ──────────────
        act_np = activations.squeeze(0).cpu().numpy()          # (C, h, w)
        act_2d = act_np.reshape(C, h * w).astype(np.float64)  # (C, H*W)

        # ── 3. Center: subtract per-channel mean ──────────────────────────────
        act_2d -= act_2d.mean(axis=1, keepdims=True)

        # ── 4. SVD on the (C x H*W) matrix ───────────────────────────────────
        #   U  : (C, C)     — left singular vectors  (channel space)
        #   S  : min(C,HW)  — singular values
        #   Vt : (C, H*W)   — right singular vectors (spatial space)
        #   The first ROW of Vt is the 1st principal component in spatial space.
        try:
            _, S, Vt = np.linalg.svd(act_2d, full_matrices=False)
        except np.linalg.LinAlgError:
            # Fallback: average pooling across channels if SVD fails
            cam_raw = act_np.mean(axis=0)
            cam_raw = np.maximum(cam_raw, 0)
            if cam_raw.max() > 0:
                cam_raw /= cam_raw.max()
            cam_np = cam_raw.astype(np.float32)
            cam_tensor = torch.from_numpy(cam_np).unsqueeze(0).unsqueeze(0)
            _, _, H_in, W_in = input_image.shape
            cam_up = F.interpolate(cam_tensor, size=(H_in, W_in),
                                   mode='bilinear', align_corners=False).squeeze().numpy()
            return cam_up.astype(np.float32), np.zeros(5)

        # Explained-variance ratios (top-5, for diagnostics)
        total_var = (S ** 2).sum() + 1e-12
        top5 = min(5, len(S))
        principal_components = (S[:top5] ** 2) / total_var

        # ── 5. First principal component → reshape to (h, w) ─────────────────
        first_pc = Vt[0].reshape(h, w)          # 1st right singular vector

        # ── 6. ReLU (keep positive activations) + normalize ──────────────────
        cam_raw = np.maximum(first_pc, 0)
        if cam_raw.max() > 0:
            cam_raw = cam_raw / cam_raw.max()

        cam_raw = cam_raw.astype(np.float32)

        # ── 7. Upsample to input spatial size ────────────────────────────────
        _, _, H_in, W_in = input_image.shape
        cam_tensor = torch.from_numpy(cam_raw).unsqueeze(0).unsqueeze(0)  # (1,1,h,w)
        cam_up = F.interpolate(cam_tensor, size=(H_in, W_in),
                               mode='bilinear', align_corners=False)
        cam_up = cam_up.squeeze().numpy()         # (H_in, W_in)

        return cam_up.astype(np.float32), principal_components


# ================== Step 7: Spatial Domain Mapping (Eigen-CAM version) ==================

def apply_eigencam_and_map_to_spatial(model, freq_image, phase, target_class, original_image):
    """
    Apply Eigen-CAM with dual-domain features and map results back to the
    spatial domain for visualization.

    Key differences from Score-CAM version:
      - Uses EigenCAM instead of ScoreCAM (no repeated forward passes).
      - cam_freq is the direct PCA-based CAM (no per-channel masking).
      - Saliency map is derived by projecting the frequency-domain CAM back
        through the inverse FFT to the spatial domain.
    """
    model.eval()
    freq_input = freq_image.clone().detach().to(device)

    # ── Generate Eigen-CAM ────────────────────────────────────────────────────
    eigencam = EigenCAM(model)
    cam_freq, principal_components = eigencam.generate_cam(freq_input)

    print(f"  Eigen-CAM: top-5 explained variance ratios = "
          f"{[f'{v:.3f}' for v in principal_components]}")

    # ── Project CAM back to spatial domain via inverse FFT ───────────────────
    freq_magnitude = freq_input[:, :3, :, :].squeeze(0).cpu().detach()

    cam_freq_tensor = torch.from_numpy(cam_freq).float()          # (H, W)
    masked_freq_magnitude = freq_magnitude * cam_freq_tensor.unsqueeze(0)  # (3, H, W)

    # Handle phase dimensions
    if phase.dim() == 4:
        phase = phase.squeeze(0)
    elif phase.dim() == 2:
        phase = phase.unsqueeze(0).repeat(3, 1, 1)

    cam_spatial = frequency_to_spatial(
        masked_freq_magnitude.unsqueeze(0),
        phase.unsqueeze(0)
    )
    cam_spatial = cam_spatial.squeeze(0)   # (3, H, W)

    # ── Build saliency map ────────────────────────────────────────────────────
    cam_spatial_abs = torch.abs(cam_spatial)
    saliency_map = torch.mean(cam_spatial_abs, dim=0).numpy()   # (H, W)

    # Invert so that high-activation regions are bright
    saliency_map = np.max(saliency_map) - saliency_map

    # Gaussian smoothing
    saliency_map = gaussian_filter(saliency_map, sigma=2.5)

    # Normalize to [0, 1]
    if saliency_map.max() > saliency_map.min():
        saliency_map = (saliency_map - saliency_map.min()) / (saliency_map.max() - saliency_map.min())
    else:
        saliency_map = np.zeros_like(saliency_map)

    # Adaptive thresholding
    threshold = np.percentile(saliency_map, 40)
    saliency_map = np.where(saliency_map > threshold, saliency_map, 0)

    # Morphological clean-up
    from scipy.ndimage import binary_closing, binary_opening
    binary_mask = saliency_map > 0
    binary_mask = binary_closing(binary_mask, structure=np.ones((5, 5)))
    binary_mask = binary_opening(binary_mask, structure=np.ones((3, 3)))
    saliency_map = saliency_map * binary_mask

    # Re-normalize
    if saliency_map.max() > 0:
        saliency_map = (saliency_map - saliency_map.min()) / (saliency_map.max() - saliency_map.min())

    # Contrast enhancement
    saliency_map = np.power(saliency_map, 0.7)

    # ── Denormalize original image for display ────────────────────────────────
    if original_image.dim() == 4:
        original_image = original_image.squeeze(0)

    mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
    std  = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
    original_denorm = torch.clamp(original_image.cpu() * std + mean, 0, 1)
    original_np = original_denorm.permute(1, 2, 0).numpy()

    # ── Overlay CAM on original image ────────────────────────────────────────
    saliency_colored = plt.cm.jet(saliency_map)[:, :, :3]
    alpha = 0.7 * saliency_map[:, :, np.newaxis]
    highlighted = np.clip((1 - alpha) * original_np + alpha * saliency_colored, 0, 1)

    return cam_spatial, saliency_map, highlighted, original_np, cam_freq, principal_components


# ================== Step 8: Visualization Functions ==================

def plot_eigencam_results(original_np, freq_magnitude, cam_freq, saliency_map,
                          highlighted, prediction, true_label, classes, confidence,
                          principal_components):
    """
    Plot comprehensive Eigen-CAM results with dual-domain info.

    Layout (2 × 4 grid):
      Row 0: Original | Frequency Spectrum | Eigen-CAM (freq) | Prediction info
      Row 1: Saliency Map | Highlighted | Heatmap Overlay | Before|After + PC bar
    """

    fig, axes = plt.subplots(2, 4, figsize=(22, 11))
    fig.suptitle(
        'Enhanced Frequency Domain CNN — Dual-Domain Feature Fusion\n'
        'Visualization: Eigen-CAM (PCA of Fused Activation Maps)',
        fontsize=15, fontweight='bold', y=1.00
    )

    # ── [0,0] Original Image ──────────────────────────────────────────────────
    axes[0, 0].imshow(original_np)
    axes[0, 0].set_title(f'Original Image\nGround Truth: {classes[true_label]}',
                         fontsize=11, fontweight='bold')
    axes[0, 0].axis('off')

    # ── [0,1] Frequency Spectrum ──────────────────────────────────────────────
    freq_display = freq_magnitude[:, :3, :, :].squeeze(0).mean(0).cpu().numpy()
    im1 = axes[0, 1].imshow(freq_display, cmap='viridis')
    axes[0, 1].set_title('Frequency Domain\n(Input Magnitude Spectrum)',
                         fontsize=11, fontweight='bold')
    axes[0, 1].axis('off')
    plt.colorbar(im1, ax=axes[0, 1], fraction=0.046, pad=0.04)

    # ── [0,2] Eigen-CAM in Frequency Domain ──────────────────────────────────
    im2 = axes[0, 2].imshow(cam_freq, cmap='jet')
    axes[0, 2].set_title('Eigen-CAM\n(1st PC of Fused Activations)', fontsize=11, fontweight='bold')
    axes[0, 2].axis('off')
    plt.colorbar(im2, ax=axes[0, 2], fraction=0.046, pad=0.04)

    # ── [0,3] Prediction ──────────────────────────────────────────────────────
    correct = "✓" if prediction == true_label else "✗"
    color   = 'green' if prediction == true_label else 'red'
    axes[0, 3].text(
        0.5, 0.5,
        f'{correct} Prediction:\n{classes[prediction]}\n\nConfidence:\n{confidence:.1f}%',
        ha='center', va='center', fontsize=13, fontweight='bold',
        bbox=dict(boxstyle='round', facecolor=color, alpha=0.3)
    )
    axes[0, 3].set_title('Model Prediction', fontsize=11, fontweight='bold')
    axes[0, 3].axis('off')

    # ── [1,0] Saliency Map ────────────────────────────────────────────────────
    im3 = axes[1, 0].imshow(saliency_map, cmap='hot')
    axes[1, 0].set_title('Saliency Map\n(Eigen-CAM → Spatial Domain)',
                         fontsize=11, fontweight='bold')
    axes[1, 0].axis('off')
    plt.colorbar(im3, ax=axes[1, 0], fraction=0.046, pad=0.04)

    # ── [1,1] Highlighted Regions ─────────────────────────────────────────────
    axes[1, 1].imshow(highlighted)
    axes[1, 1].set_title('Highlighted Regions\n(Dual-Domain Eigen-CAM)',
                         fontsize=11, fontweight='bold')
    axes[1, 1].axis('off')

    # ── [1,2] Importance Heatmap overlay ─────────────────────────────────────
    axes[1, 2].imshow(original_np)
    axes[1, 2].imshow(saliency_map, cmap='jet', alpha=0.5)
    axes[1, 2].set_title('Importance Heatmap\n(50 % Overlay)',
                         fontsize=11, fontweight='bold')
    axes[1, 2].axis('off')

    # ── [1,3] Before | After + explained-variance bar chart ──────────────────
    # Split the axes: top half = before/after image, bottom half = bar chart
    divider_y = 0.45   # fraction at which we split the axes vertically

    # Before | After image (use inset)
    ax_img = axes[1, 3].inset_axes([0, divider_y, 1, 1 - divider_y])
    ax_img.imshow(np.concatenate([original_np, highlighted], axis=1))
    ax_img.set_title('Before  |  After\n(Eigen-CAM overlay)', fontsize=9)
    ax_img.axis('off')

    # Explained-variance bar chart
    ax_bar = axes[1, 3].inset_axes([0.05, 0, 0.9, divider_y - 0.05])
    top_n = len(principal_components)
    bar_colors = ['#e63946' if i == 0 else '#457b9d' for i in range(top_n)]
    ax_bar.bar(range(1, top_n + 1), principal_components * 100, color=bar_colors)
    ax_bar.set_xlabel('Principal Component', fontsize=8)
    ax_bar.set_ylabel('Explained Var. (%)', fontsize=8)
    ax_bar.set_title('Eigen-CAM: Top-5 PC\nExplained Variance', fontsize=8)
    ax_bar.set_xticks(range(1, top_n + 1))
    ax_bar.tick_params(labelsize=7)

    # Hide the parent axes frame
    axes[1, 3].axis('off')

    plt.tight_layout()
    plt.show()


def plot_training_curves(train_losses, val_losses, train_accuracies, val_accuracies):
    """Plot training and validation curves"""
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    
    epochs = range(1, len(train_losses) + 1)
    ax1.plot(epochs, train_losses, 'b-', label='Training Loss', linewidth=2)
    ax1.plot(epochs, val_losses, 'r-', label='Validation Loss', linewidth=2)
    ax1.set_xlabel('Epoch', fontsize=12)
    ax1.set_ylabel('Loss', fontsize=12)
    ax1.set_title('Training and Validation Loss', fontsize=14, fontweight='bold')
    ax1.legend(fontsize=11)
    ax1.grid(True, alpha=0.3)
    
    ax2.plot(epochs, train_accuracies, 'b-', label='Training Accuracy', linewidth=2)
    ax2.plot(epochs, val_accuracies, 'r-', label='Validation Accuracy', linewidth=2)
    ax2.set_xlabel('Epoch', fontsize=12)
    ax2.set_ylabel('Accuracy (%)', fontsize=12)
    ax2.set_title('Training and Validation Accuracy', fontsize=14, fontweight='bold')
    ax2.legend(fontsize=11)
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

# ================== Step 9: Main Execution Pipeline ==================

def main():
    print("="*80)
    print("Enhanced Frequency Domain CNN with Dual-Domain Feature Fusion")
    print("Innovation: x_fused = x + fft(x) before GAP layer")
    print("CAM Method: Eigen-CAM (PCA of fused activation maps, gradient-free)")
    print("OPTIMIZED FOR 16GB GPU")
    print("="*80)
    
    # Dataset path - MODIFY THIS PATH
    data_root = r'/kaggle/input/fruits/fruits-360_100x100/fruits-360'
    
    # Check if path exists
    if not os.path.exists(data_root):
        print(f"\nERROR: Dataset path not found: {data_root}")
        print("Please modify the 'data_root' variable to point to your dataset location.")
        return
    
    print("\n[Step 1] Loading Fruits-360 dataset...")
    try:
        trainset, testset, classes = load_fruits_dataset(data_root)
        print(f"Number of classes: {len(classes)}")
    except Exception as e:
        print(f"ERROR loading dataset: {e}")
        return
    
    print("\n[Step 2] Splitting training set into train/validation...")
    train_size = int(0.85 * len(trainset))
    val_size = len(trainset) - train_size
    train_subset, val_subset = torch.utils.data.random_split(
        trainset, [train_size, val_size],
        generator=torch.Generator().manual_seed(42)
    )
    
    print(f"Training samples: {len(train_subset)}")
    print(f"Validation samples: {len(val_subset)}")
    print(f"Test samples: {len(testset)}")
    
    print("\n[Step 3] Converting to frequency domain using FFT...")
    freq_train_dataset = FrequencyDomainDataset(train_subset, cache_freq=False)
    freq_val_dataset = FrequencyDomainDataset(val_subset, cache_freq=False)
    freq_test_dataset = FrequencyDomainDataset(testset, cache_freq=False)
    
    # Optimized batch size and num_workers for 16GB GPU
    batch_size = 128
    num_workers = 4 if os.name != 'nt' else 0
    
    print(f"Batch size: {batch_size}")
    print(f"Num workers: {num_workers}")
    
    train_loader = DataLoader(
        freq_train_dataset, 
        batch_size=batch_size, 
        shuffle=True, 
        num_workers=num_workers, 
        pin_memory=True,
        persistent_workers=False,
        prefetch_factor=2 if num_workers > 0 else None
    )
    
    val_loader = DataLoader(
        freq_val_dataset, 
        batch_size=batch_size, 
        shuffle=False,
        num_workers=num_workers, 
        pin_memory=True,
        persistent_workers=False,
        prefetch_factor=2 if num_workers > 0 else None
    )
    
    test_loader = DataLoader(
        freq_test_dataset, 
        batch_size=1, 
        shuffle=False,
        num_workers=0
    )
    
    print("\n[Step 4] Initializing Enhanced ResNet50 with Dual-Domain Fusion...")
    model = EnhancedFrequencyDomainCNN(num_classes=len(classes), dropout_rate=0.5).to(device)
    print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")
    print("Architecture: ResNet50 + Dual-Domain Feature Fusion (x + fft(x)) before GAP")
    
    # Check GPU memory
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        print(f"GPU memory allocated: {torch.cuda.memory_allocated(0) / 1e9:.2f} GB")
        print(f"GPU memory reserved: {torch.cuda.memory_reserved(0) / 1e9:.2f} GB")
    
    print("\n[Step 5] Training model with dual-domain features...")
    print("Starting training loop...\n")
    
    try:
        train_losses, val_losses, train_accuracies, val_accuracies = train_model(
            model, train_loader, val_loader, 
            epochs=2,
            lr=0.001,
            weight_decay=5e-4
        )
    except Exception as e:
        print(f"\nERROR during training: {e}")
        import traceback
        traceback.print_exc()
        return
    
    print("\n[Step 5.1] Plotting training curves...")
    plot_training_curves(train_losses, val_losses, train_accuracies, val_accuracies)
    
    print("\n[Step 6] Evaluating on test set...")
    model.eval()
    correct = 0
    total = 0
    
    all_predictions = []
    all_labels = []
    
    with torch.no_grad():
        test_pbar = tqdm(test_loader, desc="Testing")
        for freq_images, labels, _ in test_pbar:
            freq_images, labels = freq_images.to(device), labels.to(device)
            outputs = model(freq_images)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
            
            all_predictions.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            
            test_pbar.set_postfix({'acc': f'{100 * correct / total:.2f}%'})
    
    test_accuracy = 100 * correct / total
    print(f"\nFinal Test Accuracy: {test_accuracy:.2f}%")
    
    print("\n[Step 7] Applying Eigen-CAM with dual-domain features...")
    print("Generating saliency maps using PCA of fused activation maps...\n")
    
    # Test on diverse samples
    np.random.seed(42)
    num_samples = min(5, len(testset))
    test_indices = np.random.choice(len(testset), num_samples, replace=False)
    
    for idx in test_indices:
        try:
            original_image, true_label = testset[idx]
            
            freq_features, phase, _ = spatial_to_frequency(original_image.unsqueeze(0))
            freq_input = freq_features.to(device)
            
            # Get model prediction
            model.eval()
            with torch.no_grad():
                output = model(freq_input)
                probabilities = F.softmax(output, dim=1)
                confidence, predicted = torch.max(probabilities.data, 1)
                predicted_class = predicted.item()
                confidence = confidence.item() * 100
            
            print(f"Sample {idx}:")
            print(f"  True Label: {classes[true_label]}")
            print(f"  Predicted: {classes[predicted_class]} ({confidence:.1f}% confidence)")
            
            # Apply Eigen-CAM with dual-domain features
            _, saliency_map, highlighted, original_np, cam_freq, principal_components = \
                apply_eigencam_and_map_to_spatial(
                    model, freq_input, phase.squeeze(0), predicted_class, original_image
                )
            
            # Visualize results
            plot_eigencam_results(
                original_np,
                freq_features,
                cam_freq,
                saliency_map,
                highlighted,
                predicted_class,
                true_label,
                classes,
                confidence,
                principal_components
            )
            
            # Clear GPU cache
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
            
            print()
        
        except Exception as e:
            print(f"ERROR processing sample {idx}: {e}")
            import traceback
            traceback.print_exc()
            continue
    
    print("="*80)
    print("Pipeline completed successfully!")
    print(f"Final Test Accuracy: {test_accuracy:.2f}%")
    print(f"Total Classes: {len(classes)}")
    print("="*80)
    print("\n🚀 KEY INNOVATION:")
    print("✓ Dual-Domain Feature Fusion: x_fused = α*x + β*fft(x)")
    print("✓ Applied before GAP layer to capture hidden frequency patterns")
    print("✓ Learnable fusion weights for optimal balance")
    print("✓ Channel attention for adaptive feature weighting")
    print("="*80)
    print("\nEigen-CAM Details:")
    print("✓ Gradient-free: no backward passes required")
    print("✓ Uses SVD / PCA on fused activation maps (C × H×W matrix)")
    print("✓ 1st principal component = most discriminative spatial pattern")
    print("✓ Much faster than Score-CAM (no per-channel forward passes)")
    print("✓ Visualizes explained-variance per principal component")
    print("="*80)
    print("\nOptimizations Applied:")
    print("✓ Mixed precision training (AMP)")
    print("✓ Optimized batch size (128) for 16GB GPU")
    print("✓ Gradient accumulation and clipping")
    print("✓ Memory-efficient data loading")
    print("✓ Periodic GPU cache clearing")
    print("✓ CuDNN benchmark mode enabled")
    print("="*80)
    
    # Save the model
    print("\n[Step 8] Saving trained model...")
    try:
        torch.save({
            'model_state_dict': model.state_dict(),
            'test_accuracy': test_accuracy,
            'classes': classes,
            'num_classes': len(classes)
        }, 'fruits_enhanced_dual_domain_cnn.pth')
        print("Model saved as 'fruits_enhanced_dual_domain_cnn.pth'")
    except Exception as e:
        print(f"ERROR saving model: {e}")
    
    # Final memory cleanup
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        print(f"\nFinal GPU memory allocated: {torch.cuda.memory_allocated(0) / 1e9:.2f} GB")

if __name__ == "__main__":
    main()